# **Vector Stores and retrievers**

Langchain Vector store and retriever , these abstraction supoort data retrieval from vector store for intregation with LLM workflow.

Thry are Imp for application that fetch data to be reasoned over as part of model inference, as in the case of retrieval augmented generation.

In [ ]:
!pip install langchain langchain-chroma langchain-groq langchain-huggingface

In [ ]:
import os
from dotenv import load_dotenv
load_dotenv()
from langchain_groq import ChatGroq

groq_api_key=os.getenv('GROQ_API_KEY')
HF_api_key=os.getenv('Hf_Token')

llm=ChatGroq(groq_api_key=groq_api_key, model="llama-3.1-8b-instant")
llm

ChatGroq(profile={'max_input_tokens': 131072, 'max_output_tokens': 8192, 'image_inputs': False, 'audio_inputs': False, 'video_inputs': False, 'image_outputs': False, 'audio_outputs': False, 'video_outputs': False, 'reasoning_output': False, 'tool_calling': True}, client=<groq.resources.chat.completions.Completions object at 0x7e7a5f1eb020>, async_client=<groq.resources.chat.completions.AsyncCompletions object at 0x7e7a5f1eb6e0>, model_name='llama-3.1-8b-instant', model_kwargs={}, groq_api_key=SecretStr('**********'))

In [ ]:
from langchain_core.documents import Document

documents=[
Document(
  page_content="Predicted Performance of Multilayered Metal-Mesh Screens. SPE Drilling & Completion. SPE-178955-PA.",
  metadata={
    "page": 0,
    "source": "C:\\Users\\username\\Desktop\\resumes_dataset\\50328713.pdf"
  }
) ,

Document(
  page_content="id: 1, name: Laptop, price: 1200",
  metadata={
    "category": "Electronics",
    "source": "products.csv"
  }
) ,

Document(
  page_content="This is a super-customized document",
  metadata={
    "file_name": "super_secret_document.txt",
    "category": "finance",
    "author": "LlamaIndex"
  }
) ]

documents

[Document(metadata={'page': 0, 'source': 'C:\\Users\\username\\Desktop\\resumes_dataset\\50328713.pdf'}, page_content='Predicted Performance of Multilayered Metal-Mesh Screens. SPE Drilling & Completion. SPE-178955-PA.'),
 Document(metadata={'category': 'Electronics', 'source': 'products.csv'}, page_content='id: 1, name: Laptop, price: 1200'),
 Document(metadata={'file_name': 'super_secret_document.txt', 'category': 'finance', 'author': 'LlamaIndex'}, page_content='This is a super-customized document')]

In [ ]:
#embedding models
from langchain_huggingface import HuggingFaceEmbeddings

embeddings=HuggingFaceEmbeddings(model="all-MiniLm-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
#vectorstore
from langchain_chroma import Chroma

vectorstores=Chroma.from_documents(documents,embedding =embeddings )
vectorstores

In [ ]:
# now we willl search in our database
vectorstores.similarity_search('price')

[Document(id='859e84c3-6375-416f-8063-17a9f91159e1', metadata={'category': 'Electronics', 'source': 'products.csv'}, page_content='id: 1, name: Laptop, price: 1200'),
 Document(id='c43dab6d-d56b-4192-8707-db911c781cae', metadata={'category': 'finance', 'author': 'LlamaIndex', 'file_name': 'super_secret_document.txt'}, page_content='This is a super-customized document'),
 Document(id='fdf157b9-e719-47c4-a86f-e2b7eff31c7e', metadata={'source': 'C:\\Users\\username\\Desktop\\resumes_dataset\\50328713.pdf', 'page': 0}, page_content='Predicted Performance of Multilayered Metal-Mesh Screens. SPE Drilling & Completion. SPE-178955-PA.')]

In [ ]:
vectorstores.similarity_search_with_score('price')

[(Document(id='859e84c3-6375-416f-8063-17a9f91159e1', metadata={'source': 'products.csv', 'category': 'Electronics'}, page_content='id: 1, name: Laptop, price: 1200'),
  0.9466290473937988),
 (Document(id='c43dab6d-d56b-4192-8707-db911c781cae', metadata={'category': 'finance', 'file_name': 'super_secret_document.txt', 'author': 'LlamaIndex'}, page_content='This is a super-customized document'),
  1.6469721794128418),
 (Document(id='fdf157b9-e719-47c4-a86f-e2b7eff31c7e', metadata={'source': 'C:\\Users\\username\\Desktop\\resumes_dataset\\50328713.pdf', 'page': 0}, page_content='Predicted Performance of Multilayered Metal-Mesh Screens. SPE Drilling & Completion. SPE-178955-PA.'),
  1.8350564241409302)]

Retrievers

LangChain Retrievers are Runnables , so they implement a standard set of method sunchrous ,asynchronous and batch operation and are designed to be incorporated in LCEL chains .

In [ ]:
# from typing import List
from langchain_core.documents import Document
from langchain_core.runnables import RunnableLambda

retrievers=RunnableLambda(vectorstores.similarity_search_with_score).bind(k=1)
retrievers.batch(["price","screen"])

[[(Document(id='859e84c3-6375-416f-8063-17a9f91159e1', metadata={'category': 'Electronics', 'source': 'products.csv'}, page_content='id: 1, name: Laptop, price: 1200'),
   0.9466290473937988)],
 [(Document(id='fdf157b9-e719-47c4-a86f-e2b7eff31c7e', metadata={'page': 0, 'source': 'C:\\Users\\username\\Desktop\\resumes_dataset\\50328713.pdf'}, page_content='Predicted Performance of Multilayered Metal-Mesh Screens. SPE Drilling & Completion. SPE-178955-PA.'),
   1.2921510934829712)]]

Vectorstores implement an as_retriever method that will generate a Retriver , specifically a VectorStoreRetriever .These retriever include specific serach_typpe and search_kwargs attribute that identify what method of underlying vector store to call and how to parameterize them .

In [ ]:
retriever=vectorstores.as_retriever(
    search_type="similarity",
    search_kwargs={"k":1 }
)
retriever.batch(["price","screen"])

[[Document(id='859e84c3-6375-416f-8063-17a9f91159e1', metadata={'source': 'products.csv', 'category': 'Electronics'}, page_content='id: 1, name: Laptop, price: 1200')],
 [Document(id='fdf157b9-e719-47c4-a86f-e2b7eff31c7e', metadata={'page': 0, 'source': 'C:\\Users\\username\\Desktop\\resumes_dataset\\50328713.pdf'}, page_content='Predicted Performance of Multilayered Metal-Mesh Screens. SPE Drilling & Completion. SPE-178955-PA.')]]

In [ ]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.runnables import RunnablePassthrough

message="""
Annwer this question using previous context only .

{question}

context:
{context}
"""

prompt=ChatPromptTemplate.from_messages([('human',message)])
rag_chain={"context":retriever,'question':RunnablePassthrough()}|prompt|llm
response=rag_chain.invoke("what compiler am i using today?")
print(response)

content='Unfortunately, I cannot determine the compiler you are using today based on the provided context. The context appears to be a document with metadata and content, but it does not contain any information about a compiler.' additional_kwargs={} response_metadata={'token_usage': {'completion_tokens': 41, 'prompt_tokens': 114, 'total_tokens': 155, 'completion_time': 0.089822088, 'completion_tokens_details': None, 'prompt_time': 0.009847374, 'prompt_tokens_details': None, 'queue_time': 0.034535056, 'total_time': 0.099669462}, 'model_name': 'llama-3.1-8b-instant', 'system_fingerprint': 'fp_020e283281', 'service_tier': 'on_demand', 'finish_reason': 'stop', 'logprobs': None, 'model_provider': 'groq'} id='lc_run--019e5ecb-1907-7ef2-bdeb-ba0722959e66-0' tool_calls=[] invalid_tool_calls=[] usage_metadata={'input_tokens': 114, 'output_tokens': 41, 'total_tokens': 155}


In [ ]:
from groq import Groq

client = Groq(
    api_key=groq_api_key,
)

models = client.models.list()

print("Available Groq Models:")
for model in models.data:
    print(f"- {model.id}")

Available Groq Models:
- allam-2-7b
- groq/compound-mini
- canopylabs/orpheus-v1-english
- openai/gpt-oss-20b
- whisper-large-v3-turbo
- meta-llama/llama-prompt-guard-2-86m
- qwen/qwen3-32b
- openai/gpt-oss-120b
- llama-3.3-70b-versatile
- whisper-large-v3
- meta-llama/llama-4-scout-17b-16e-instruct
- canopylabs/orpheus-arabic-saudi
- meta-llama/llama-prompt-guard-2-22m
- llama-3.1-8b-instant
- groq/compound
- openai/gpt-oss-safeguard-20b
